In [0]:
# ================================================================
# Customer Offer Optimisation under Uncertainty
# ================================================================
# Propensity scoring, Monte Carlo simulation, 
# LP optimisation, Holdout measurement
#
# Author: Mina Rezaei
# Domain: Digital marketing analytics
# ================================================================

import numpy as np
import pandas as pd
import json
import warnings
from scipy.optimize import linprog
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import mlflow
import mlflow.sklearn

warnings.filterwarnings("ignore")
np.random.seed(42)

print("Libraries loaded ✓")

Libraries loaded ✓


In [0]:
# ================================================================
# STEP 2 — Synthetic Customer Data
# ================================================================
# Simulates a customer base for a digital marketing platform.
# In production this would read from a Delta table e.g.:
# df = spark.read.table("dcoe.customer_features").toPandas()
# ================================================================

OFFER_TYPES = ["BOGO", "Discount_20", "Free_Item", "Loyalty_Points", "Bundle"]
OFFER_COSTS = {"BOGO": 0.35, "Discount_20": 0.20, "Free_Item": 0.45,
"Loyalty_Points": 0.10, "Bundle": 0.30}
OFFER_REVENUE = {"BOGO": 8.0, "Discount_20": 5.5, "Free_Item": 4.0,
"Loyalty_Points": 3.0, "Bundle": 7.0}

def generate_customer_data(n=5000):
    np.random.seed(42)

    segments = np.random.choice(
        ["Frequent", "Lapsed", "New", "High-Value", "Price-Sensitive"],
        size=n, p=[0.25, 0.20, 0.20, 0.15, 0.20])

    days_since = np.where(segments=="Lapsed", np.random.randint(60,180,n),
        np.where(segments=="Frequent", np.random.randint(1,14,n),
        np.where(segments=="New", np.random.randint(1,30,n),
        np.where(segments=="High-Value",np.random.randint(3,21,n),
        np.random.randint(7,60,n)))))

    orders_90d = np.where(segments=="Frequent", np.random.randint(8,20,n),
        np.where(segments=="Lapsed", np.random.randint(0,3,n),
        np.where(segments=="New", np.random.randint(1,4,n),
        np.where(segments=="High-Value", np.random.randint(5,15,n),
        np.random.randint(2,8,n)))))

    avg_val = np.where(segments=="High-Value", np.random.normal(28,5,n),
        np.where(segments=="Price-Sensitive", np.random.normal(12,3,n),
        np.where(segments=="Frequent", np.random.normal(18,4,n),
        np.random.normal(16,5,n)))).clip(5,60)

    app_opens = np.where(segments=="Frequent", np.random.poisson(15,n),
        np.where(segments=="Lapsed", np.random.poisson(1,n),
        np.random.poisson(6,n)))

    push_ctr = np.random.beta(2,8,n).clip(0,1)
    loyalty_tier = np.random.choice(["Bronze","Silver","Gold","Platinum"],n,p=[0.4,0.3,0.2,0.1])
    market = np.random.choice(["NZ","AU","SG","JP","UK"],n,p=[0.25,0.25,0.2,0.15,0.15])
    is_weekend = np.random.binomial(1,0.4,n)
    offer_type = np.random.choice(OFFER_TYPES,n)

    base_prob = (
        0.15
        + 0.25*(segments=="Frequent") + 0.15*(segments=="High-Value")
        - 0.10*(segments=="Lapsed")
        + 0.05*(loyalty_tier=="Gold") + 0.10*(loyalty_tier=="Platinum")
        + 0.08*(offer_type=="Free_Item") + 0.06*(offer_type=="BOGO")
        - 0.05*(offer_type=="Loyalty_Points")
        + 0.003*app_opens - 0.001*days_since + 0.01*orders_90d
        + np.random.normal(0,0.05,n)
    ).clip(0.01,0.95)

    return pd.DataFrame({
        "customer_id": [f"CUST_{i:05d}" for i in range(n)],
        "segment": segments,
        "days_since_last_order": days_since,
        "orders_last_90d": orders_90d,
        "avg_order_value": avg_val.round(2),
        "app_opens_last_30d": app_opens,
        "push_notif_ctr": push_ctr.round(4),
        "loyalty_tier": loyalty_tier,
        "market": market,
        "is_weekend_user": is_weekend,
        "offer_type": offer_type,
        "redeemed": np.random.binomial(1, base_prob, n),
        "true_redemption_prob": base_prob.round(4)
    })

# Generate and preview
df = generate_customer_data(5000)

print(f"Customers: {len(df):,}")
print(f"Redemption rate: {df['redeemed'].mean():.1%}")
print(f"Segments: {df['segment'].value_counts().to_dict()}")
print()
display(df.head())

Customers: 5,000
Redemption rate: 32.8%
Segments: {'Frequent': 1283, 'Price-Sensitive': 999, 'Lapsed': 994, 'New': 984, 'High-Value': 740}



customer_id,segment,days_since_last_order,orders_last_90d,avg_order_value,app_opens_last_30d,push_notif_ctr,loyalty_tier,market,is_weekend_user,offer_type,redeemed,true_redemption_prob
CUST_00000,Lapsed,110,1,18.37,2,0.104,Silver,SG,1,Loyalty_Points,0,0.01
CUST_00001,Price-Sensitive,25,2,13.44,4,0.157,Bronze,SG,1,BOGO,0,0.2448
CUST_00002,High-Value,12,14,33.3,4,0.153,Gold,SG,0,Discount_20,1,0.4682
CUST_00003,New,7,2,10.24,7,0.1526,Gold,JP,1,BOGO,1,0.2485
CUST_00004,Frequent,13,15,29.54,12,0.2656,Gold,AU,0,Loyalty_Points,1,0.4355


In [0]:
# ================================================================
# STEP 3 — Feature Engineering
# ================================================================
# We create new features that better capture customer behaviour.
# Raw numbers alone (e.g. days_since_last_order) are less powerful than transformed signals the model can learn from more easily.
# ================================================================

def engineer_features(df):
    df = df.copy()

    # How "fresh" is this customer?
    # A customer who ordered 3 days ago scores ~0.90
    # A customer who ordered 60 days ago scores ~0.14
    # The longer they've been away, the closer to zero
    df["recency_score"] = np.exp(-df["days_since_last_order"] / 30)

    # How engaged is this customer with the app?
    # Combines how often they open the app AND whether they click notifications
    # A customer who opens the app 15x AND clicks 30% of notifications
    # scores much higher than one who opens once and never clicks
    df["engagement_score"] = df["app_opens_last_30d"] * df["push_notif_ctr"]

    # How valuable is this customer overall?
    # Combines how much they spend AND how often they order
    # A customer spending $28 avg across 14 orders = 392
    # A customer spending $12 avg across 2 orders = 24
    df["value_frequency"] = df["avg_order_value"] * df["orders_last_90d"]

    # Convert text columns into numbers the model can understand
    # e.g. "Frequent" becomes a column with 1/0
    # e.g. "Gold" becomes a column with 1/0
    # This is called one-hot encoding
    df = pd.get_dummies(
        df,
        columns=["segment", "loyalty_tier", "market", "offer_type"],
        drop_first=False
    )

    # These are the columns the model will learn from
    # We exclude customer_id (just a label)
    # We exclude redeemed (that's what we're trying to predict)
    # We exclude true_redemption_prob (that's the answer we're hiding from the model)
    feature_cols = [
        c for c in df.columns
        if c not in ["customer_id", "redeemed", "true_redemption_prob"]
    ]

    return df, feature_cols

# Run it
df_feat, feature_cols = engineer_features(df)

print(f"Original columns: {len(df.columns)}")
print(f"After engineering: {len(df_feat.columns)}")
print(f"Features for model: {len(feature_cols)}")
print()
print("New features created:")
print(" ✓ recency_score")
print(" ✓ engagement_score")
print(" ✓ value_frequency")
print()
print("Sample feature names:")
print([c for c in feature_cols if any(
    x in c for x in ["segment","loyalty","recency","engagement","value"]
)][:10])

Original columns: 13
After engineering: 31
Features for model: 28

New features created:
 ✓ recency_score
 ✓ engagement_score
 ✓ value_frequency

Sample feature names:
['avg_order_value', 'recency_score', 'engagement_score', 'value_frequency', 'segment_Frequent', 'segment_High-Value', 'segment_Lapsed', 'segment_New', 'segment_Price-Sensitive', 'loyalty_tier_Bronze']


In [0]:
# ================================================================
# STEP 4 — Train Predictive Model (Gradient Boosting)
# ================================================================
# We train a model to predict redemption probability per customer.
# In Mercury's context this is the "propensity score" —
# the likelihood a customer accepts a given offer.
#
# We use Gradient Boosting because:
# - Handles mixed data types well (numbers + one-hot columns)
# - Naturally captures non-linear relationships
# - Gives us feature importance for explainability
# - Same family as XGBoost which you already know from Plexure
# ================================================================

def train_model(df_feat, feature_cols):
    # X = the features the model learns FROM (the 28 columns)
    # y = what we're trying to predict (did they redeem? 1 or 0)
    X = df_feat[feature_cols].astype(float)
    y = df_feat["redeemed"]

    # Split into training set (80%) and test set (20%)
    # Training set = what the model learns from
    # Test set = what we use to check if it learned correctly
    # We never let the model see the test set during training
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y # ensures same redemption ratio in both sets
    )

    # Define the model
    model = GradientBoostingClassifier(
        n_estimators=150, # number of trees to build
        max_depth=4, # how deep each tree can go
        learning_rate=0.05, # how much each tree corrects the previous one
        subsample=0.8, # use 80% of data per tree (reduces overfitting)
        random_state=42
    )

    # Train it — this is where the learning happens
    print("Training model...")
    model.fit(X_train, y_train)

    # Evaluate on the test set
    y_prob = model.predict_proba(X_test)[:, 1] # probability of redemption
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    print(f" ROC-AUC : {auc:.4f} (1.0 = perfect, 0.5 = random guessing)")
    print(f" PR-AUC : {pr_auc:.4f} (better metric for imbalanced classes)")

    return model, X_train, X_test, y_train, y_test, y_prob, auc, pr_auc

# Run it
model, X_train, X_test, y_train, y_test, y_prob, auc, pr_auc = train_model(
    df_feat, feature_cols
)

print()
print(f"Training set size : {len(X_train):,} customers")
print(f"Test set size : {len(X_test):,} customers")
print()
print("Model trained ✓")

Training model...
 ROC-AUC : 0.7631 (1.0 = perfect, 0.5 = random guessing)
 PR-AUC : 0.5544 (better metric for imbalanced classes)

Training set size : 4,000 customers
Test set size : 1,000 customers

Model trained ✓


In [0]:
# ================================================================
# STEP 5 — Monte Carlo Simulation
# ================================================================
# The model gives us ONE predicted probability per customer.
# But that prediction is never perfectly accurate — there is
# always uncertainty in real customer behaviour.
#
# Monte Carlo says: instead of trusting one number, let's run
# 1,000 simulated scenarios with slight random variation each
# time, and see the full range of possible profit outcomes.
#
# This is what separates a decision science pipeline from
# a basic ML model — we quantify RISK, not just expectation.
# ================================================================

def monte_carlo_simulation(model, X_candidates, n_simulations=1000):
    # Get the model's predicted probability for each customer
    # This is the "base" — our best guess before adding uncertainty
    base_probs = model.predict_proba(X_candidates)[:, 1]

    print(f"Average predicted redemption probability: {base_probs.mean():.1%}")
    print(f"Running {n_simulations:,} simulations per offer type...")
    print()

    results = {}

    for offer in OFFER_TYPES:
        cost = OFFER_COSTS[offer]
        revenue = OFFER_REVENUE[offer]

        simulated_profits = []

        for _ in range(n_simulations):
            # Add a tiny random noise to each customer's probability
            # This simulates real-world uncertainty —
            # our model is never 100% right about every customer
            noise = np.random.normal(0, 0.03, len(base_probs))
            sim_probs = (base_probs + noise).clip(0.01, 0.99)

            # Simulate whether each customer actually redeems
            # (flip a weighted coin for each customer)
            redemptions = np.random.binomial(1, sim_probs)

            # Calculate average profit for this scenario
            # Each customer either redeems (revenue - cost) or doesn't (-cost)
            profit = (redemptions * revenue - cost).mean()
            simulated_profits.append(profit)

        simulated_profits = np.array(simulated_profits)

        results[offer] = {
            "mean_profit": float(simulated_profits.mean()),
            "std": float(simulated_profits.std()),
            "ci_lower": float(np.percentile(simulated_profits, 5)),
            "ci_upper": float(np.percentile(simulated_profits, 95)),
            "prob_positive": float((simulated_profits > 0).mean())
        }

    return results, base_probs

# Run it
mc_results, base_probs = monte_carlo_simulation(model, X_test)

# Print results nicely
print(f"{'Offer':<18} {'E[Profit]':>10} {'CI Lower':>10} {'CI Upper':>10} {'P(Profit>0)':>12}")
print("-" * 65)
for offer, r in mc_results.items():
    print(
        f"{offer:<18} "
        f"${r['mean_profit']:>8.3f} "
        f"${r['ci_lower']:>8.3f} "
        f"${r['ci_upper']:>8.3f} "
        f"{r['prob_positive']:>11.1%}"
    )

print()
print("Monte Carlo simulation complete ✓")

Average predicted redemption probability: 31.1%
Running 1,000 simulations per offer type...

Offer               E[Profit]   CI Lower   CI Upper  P(Profit>0)
-----------------------------------------------------------------
BOGO               $   2.143 $   1.978 $   2.314      100.0%
Discount_20        $   1.517 $   1.401 $   1.626      100.0%
Free_Item          $   0.799 $   0.718 $   0.886      100.0%
Loyalty_Points     $   0.834 $   0.776 $   0.893      100.0%
Bundle             $   1.878 $   1.723 $   2.024      100.0%

Monte Carlo simulation complete ✓


In [0]:
# ================================================================
# STEP 6 — Linear Programming Optimisation
# ================================================================
# Given a fixed budget and a set of customers to target,
# find the optimal fraction of customers to receive each offer
# to maximise total expected profit.
#
# Decision variables: x[i] = fraction of customers getting offer i
# Objective: maximise sum(net_value[i] * x[i] * n_customers)
# Constraint: sum(cost[i] * x[i] * n_customers) <= budget
#
# This directly maps to Mercury's FY26 goal:
# "Sales Efficiency Uplift" — get maximum return per dollar spent
# ================================================================

def optimise_offer_allocation(base_probs, budget=5000, n_customers=500):
    revenues = [OFFER_REVENUE[o] for o in OFFER_TYPES]
    costs = [OFFER_COSTS[o] for o in OFFER_TYPES]

    # Expected net value per customer per offer
    # = (probability of redemption × revenue) - cost
    avg_prob = base_probs.mean()
    net_vals = [avg_prob * rev - cost for rev, cost in zip(revenues, costs)]

    print("Expected net value per customer per offer:")
    for offer, val in zip(OFFER_TYPES, net_vals):
        print(f" {offer:<18} ${val:.3f}")
    print()

    # Linear programming setup
    # linprog MINIMISES — so we negate to effectively MAXIMISE
    c = [-v * n_customers for v in net_vals]

    # Budget constraint: total spend cannot exceed budget
    A_ub = [[cost * n_customers for cost in costs]]
    b_ub = [budget]

    # Each offer fraction must be between 0 and 1
    # (0% to 100% of customers)
    bounds = [(0, 1) for _ in OFFER_TYPES]

    # Solve it
    result = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=bounds,
        method="highs"
    )

    allocation = {o: round(x, 4) for o, x in zip(OFFER_TYPES, result.x)}
    expected_value = float(-result.fun) # negate back to positive

    return allocation, expected_value

# Run it
allocation, expected_value = optimise_offer_allocation(
    base_probs,
    budget=5000,
    n_customers=500
)

# Print results
print("=" * 50)
print("OPTIMAL OFFER ALLOCATION")
print("=" * 50)
print(f"Budget: $5,000")
print(f"Customers: 500")
print()
print(f"{'Offer':<20} {'Allocation':>12} {'Customers':>12}")
print("-" * 46)
for offer, frac in allocation.items():
    n = round(frac * 500)
    print(f"{offer:<20} {frac*100:>11.1f}% {n:>11,}")

print()
print(f"Expected total value: ${expected_value:,.2f}")
print()
print("Optimisation complete ✓")

Expected net value per customer per offer:
 BOGO               $2.138
 Discount_20        $1.510
 Free_Item          $0.794
 Loyalty_Points     $0.833
 Bundle             $1.877

OPTIMAL OFFER ALLOCATION
Budget: $5,000
Customers: 500

Offer                  Allocation    Customers
----------------------------------------------
BOGO                       100.0%         500
Discount_20                100.0%         500
Free_Item                  100.0%         500
Loyalty_Points             100.0%         500
Bundle                     100.0%         500

Expected total value: $3,575.53

Optimisation complete ✓


In [0]:
# Re-run with more realistic constraints
allocation, expected_value = optimise_offer_allocation(
    base_probs,
    budget=500, # tighter budget
    n_customers=500 # same customers
)

print("=" * 50)
print("OPTIMAL OFFER ALLOCATION (tight budget)")
print("=" * 50)
print(f"Budget: $500")
print(f"Customers: 500")
print()
print(f"{'Offer':<20} {'Allocation':>12} {'Customers':>12}")
print("-" * 46)
for offer, frac in allocation.items():
    n = round(frac * 500)
    print(f"{offer:<20} {frac*100:>11.1f}% {n:>11,}")

print()
print(f"Expected total value: ${expected_value:,.2f}")
print()
print("Optimisation complete ✓")

Expected net value per customer per offer:
 BOGO               $2.138
 Discount_20        $1.510
 Free_Item          $0.794
 Loyalty_Points     $0.833
 Bundle             $1.877

OPTIMAL OFFER ALLOCATION (tight budget)
Budget: $500
Customers: 500

Offer                  Allocation    Customers
----------------------------------------------
BOGO                       100.0%         500
Discount_20                100.0%         500
Free_Item                   11.1%          56
Loyalty_Points             100.0%         500
Bundle                     100.0%         500

Expected total value: $3,222.73

Optimisation complete ✓


In [0]:
# ================================================================
# STEP 7 — Propensity Scoring Output Table
# ================================================================
# This is the production output of the pipeline.
# In Mercury's architecture this table would be written to a
# Databricks Delta table and read by Salesforce Data Cloud
# via BYOL (Bring Your Own Lake) zero-copy federation.
#
# The Journey Builder Decision Split would then route customers
# based on their tier:
# High → richer offer arm (e.g. BOGO or Bundle)
# Med → standard nurture
# Low → suppress or nurture-only
#
# Schema mirrors Mercury DSCoE pilot spec (section 9.1):
# customer_id | propensity_score | score_date | model_version | tier
# ================================================================

from datetime import date

def generate_scoring_output(model, df_feat, df_original, feature_cols):
    # Get propensity score for every customer
    X_all = df_feat[feature_cols].astype(float)
    scores = model.predict_proba(X_all)[:, 1]

    # Assign tier based on score thresholds
    # High = top 30% most likely to accept
    # Med = middle 40%
    # Low = bottom 30% least likely to accept
    high_threshold = np.percentile(scores, 70) # top 30%
    low_threshold = np.percentile(scores, 30) # bottom 30%

    tiers = np.where(
        scores >= high_threshold, "High",
        np.where(scores >= low_threshold, "Med", "Low")
    )

    # Build the output table
    scoring_output = pd.DataFrame({
        "customer_id": df_original["customer_id"],
        "propensity_score": scores.round(4),
        "score_date": date.today().isoformat(),
        "model_version": "gradient_boosting_v1",
        "tier": tiers
    })

    return scoring_output, high_threshold, low_threshold

# Run it
scoring_output, high_thresh, low_thresh = generate_scoring_output(
    model, df_feat, df, feature_cols
)

# Summary
print("=" * 55)
print("PROPENSITY SCORING OUTPUT")
print("=" * 55)
print(f"Total customers scored: {len(scoring_output):,}")
print()
print(f"Score thresholds:")
print(f" High tier (>=): {high_thresh:.4f}")
print(f" Low tier (<): {low_thresh:.4f}")
print()

tier_counts = scoring_output["tier"].value_counts()
for tier in ["High", "Med", "Low"]:
    n = tier_counts.get(tier, 0)
    pct = n / len(scoring_output) * 100
    print(f" {tier} tier: {n:,} customers ({pct:.0f}%)")

print()
print("Sample output (first 10 rows):")
display(scoring_output.head(10))

print()
print("In production this would be written to Delta table:")
print(" spark.createDataFrame(scoring_output)")
print(" .write.mode('overwrite')")
print(" .saveAsTable('dcoe.propensity_scores_offer')")
print()
print("Scoring output complete ✓")

PROPENSITY SCORING OUTPUT
Total customers scored: 5,000

Score thresholds:
 High tier (>=): 0.4811
 Low tier (<): 0.1628

 High tier: 1,500 customers (30%)
 Med tier: 2,000 customers (40%)
 Low tier: 1,500 customers (30%)

Sample output (first 10 rows):


customer_id,propensity_score,score_date,model_version,tier
CUST_00000,0.0365,2026-05-23,gradient_boosting_v1,Low
CUST_00001,0.2259,2026-05-23,gradient_boosting_v1,Med
CUST_00002,0.6966,2026-05-23,gradient_boosting_v1,High
CUST_00003,0.2624,2026-05-23,gradient_boosting_v1,Med
CUST_00004,0.7116,2026-05-23,gradient_boosting_v1,High
CUST_00005,0.5658,2026-05-23,gradient_boosting_v1,High
CUST_00006,0.5603,2026-05-23,gradient_boosting_v1,High
CUST_00007,0.2922,2026-05-23,gradient_boosting_v1,Med
CUST_00008,0.5568,2026-05-23,gradient_boosting_v1,High
CUST_00009,0.5227,2026-05-23,gradient_boosting_v1,High



In production this would be written to Delta table:
 spark.createDataFrame(scoring_output)
 .write.mode('overwrite')
 .saveAsTable('dcoe.propensity_scores_offer')

Scoring output complete ✓


In [0]:
# ================================================================
# STEP 8 — UCG Holdout Group
# ================================================================
# UCG = Universal Control Group
# Mercury already runs a UCG methodology (customer_ucg flag).
# Any new experiment must respect existing UCG assignment.
#
# For our pilot we:
# 1. Take all High tier customers
# 2. Randomly assign 10% to holdout (get baseline treatment)
# 3. Remaining 90% get the model-driven offer
# 4. After campaign: compare conversion rates between groups
# 5. The difference = incremental lift from the model
#
# This is the measurement layer that proves business value.
# Without it Mercury cannot know if the model helped or not.
# ================================================================

import hashlib

def create_ucg_holdout(scoring_output, holdout_pct=0.10, random_seed=42):
 
 # Only apply holdout to High tier customers
 # These are the ones entering the "richer offer arm"
 # Med and Low tiers follow standard journey logic unchanged
 high_tier = scoring_output[
 scoring_output["tier"] == "High"
 ].copy()
 
 med_low_tier = scoring_output[
 scoring_output["tier"] != "High"
 ].copy()
 
 # Randomly assign holdout flag
 # Use random_seed so the assignment is reproducible
 # Same customer always gets same assignment across runs
 np.random.seed(random_seed)
 n_high = len(high_tier)
 n_holdout = int(n_high * holdout_pct)
 
 holdout_indices = np.random.choice(
 high_tier.index,
 size=n_holdout,
 replace=False
 )
 
 high_tier["ucg_holdout"] = 0 # 0 = treatment group (gets model offer)
 high_tier.loc[holdout_indices, "ucg_holdout"] = 1 # 1 = holdout (baseline)
 
 # Med and Low tier customers are not in the experiment
 med_low_tier["ucg_holdout"] = -1 # -1 = not in experiment
 
 # Combine back together
 full_output = pd.concat([high_tier, med_low_tier]).sort_index()
 
 # Add treatment arm label — useful for Salesforce journey routing
 full_output["treatment_arm"] = full_output.apply(
 lambda row: 
 "holdout" if row["ucg_holdout"] == 1
 else "model_driven" if row["ucg_holdout"] == 0
 else "standard_nurture" if row["tier"] == "Med"
 else "suppress",
 axis=1
 )
 
 return full_output, n_holdout

# Run it
full_output, n_holdout = create_ucg_holdout(scoring_output)

# Summary
print("=" * 55)
print("UCG HOLDOUT ASSIGNMENT")
print("=" * 55)
print()

arm_counts = full_output["treatment_arm"].value_counts()
for arm in ["model_driven", "holdout", "standard_nurture", "suppress"]:
 n = arm_counts[arm]
 pct = n / len(full_output) * 100
 print(f" {arm:<20} {n:>5,} customers ({pct:.0f}%)")

print()
print("High tier breakdown:")
high = full_output[full_output["tier"] == "High"]
print(f" Treatment (model_driven): "
 f"{(high['ucg_holdout']==0).sum():,} customers")
print(f" Holdout (baseline): "
 f"{(high['ucg_holdout']==1).sum():,} customers")

print()

# Simulate what happens AFTER the campaign
# In reality Mercury would observe real conversion events
# Here we simulate based on true_redemption_prob
print("=" * 55)
print("SIMULATED CAMPAIGN RESULTS")
print("=" * 55)
print("(In production these would be real Salesforce conversion events)")
print()

# Merge true probabilities back in for simulation
full_sim = full_output.merge(
 df[["customer_id", "true_redemption_prob"]],
 on="customer_id"
)

# Treatment group gets model-driven offer (slight boost)
# Holdout gets baseline (no boost — standard treatment)
treatment = full_sim[full_sim["treatment_arm"] == "model_driven"]
holdout = full_sim[full_sim["treatment_arm"] == "holdout"]

# Simulate conversions
np.random.seed(42)
treatment_conversions = np.random.binomial(
 1, 
 (treatment["true_redemption_prob"] + 0.05).clip(0, 1)
)
holdout_conversions = np.random.binomial(
 1,
 holdout["true_redemption_prob"]
)

treatment_rate = treatment_conversions.mean()
holdout_rate = holdout_conversions.mean()
lift = treatment_rate - holdout_rate
lift_pct = (lift / holdout_rate) * 100

print(f" Treatment group conversion rate: {treatment_rate:.1%}")
print(f" Holdout group conversion rate: {holdout_rate:.1%}")
print(f" Absolute lift: +{lift:.1%}")
print(f" Relative lift: +{lift_pct:.1f}%")
print()

# Revenue impact
avg_revenue = np.mean(list(OFFER_REVENUE.values()))
extra_converts = int(lift * len(treatment))
revenue_impact = extra_converts * avg_revenue

print(f" Extra conversions from model: {extra_converts:,}")
print(f" Estimated revenue impact: ${revenue_impact:,.2f}")
print()
print("UCG holdout complete ✓")
print()
print("Full output table ready for Salesforce Data Cloud:")
display(full_output[[
 "customer_id","propensity_score","tier",
 "ucg_holdout","treatment_arm"
]].head(10))

UCG HOLDOUT ASSIGNMENT

 model_driven         1,350 customers (27%)
 holdout                150 customers (3%)
 standard_nurture     2,000 customers (40%)
 suppress             1,500 customers (30%)

High tier breakdown:
 Treatment (model_driven): 1,350 customers
 Holdout (baseline): 150 customers

SIMULATED CAMPAIGN RESULTS
(In production these would be real Salesforce conversion events)

 Treatment group conversion rate: 63.8%
 Holdout group conversion rate: 51.3%
 Absolute lift: +12.4%
 Relative lift: +24.2%

 Extra conversions from model: 168
 Estimated revenue impact: $924.00

UCG holdout complete ✓

Full output table ready for Salesforce Data Cloud:


customer_id,propensity_score,tier,ucg_holdout,treatment_arm
CUST_00000,0.0365,Low,-1,suppress
CUST_00001,0.2259,Med,-1,standard_nurture
CUST_00002,0.6966,High,0,model_driven
CUST_00003,0.2624,Med,-1,standard_nurture
CUST_00004,0.7116,High,0,model_driven
CUST_00005,0.5658,High,0,model_driven
CUST_00006,0.5603,High,0,model_driven
CUST_00007,0.2922,Med,-1,standard_nurture
CUST_00008,0.5568,High,0,model_driven
CUST_00009,0.5227,High,0,model_driven


In [0]:
# ================================================================
# STEP 9 — MLflow Experiment Tracking
# ================================================================
# MLflow records every run of this pipeline so we can:
# - Compare experiments (e.g. budget=$500 vs budget=$5,000)
# - Reproduce any past result exactly
# - Register the model for production deployment
# - Share results with the team without re-running anything
#
# In Mercury's Databricks workspace this would be visible to
# the entire DSCoE team in the MLflow UI (Experiments tab)
# ================================================================

# Set the experiment name
# All runs of this pipeline will be grouped under this name
mlflow.set_experiment("/customer-offer-optimisation")

# ── RUN 1: Generous budget ($5,000) ─────────────────────────────
print("Logging Run 1: generous budget ($5,000)...")

with mlflow.start_run(run_name="budget_5000"):

    # LOG PARAMETERS
    # These are the settings that went into this experiment
    mlflow.log_params({
        "n_customers": 5000,
        "test_size": 0.2,
        "n_estimators": 150,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "mc_simulations": 1000,
        "opt_budget": 5000,
        "opt_n_targets": 500,
        "holdout_pct": 0.10,
        "random_seed": 42
    })

    # LOG METRICS
    # These are the results — what we compare across runs
    mlflow.log_metric("roc_auc", round(auc, 4))
    mlflow.log_metric("pr_auc", round(pr_auc, 4))
    mlflow.log_metric("redemption_rate", round(df["redeemed"].mean(), 4))
    mlflow.log_metric("opt_expected_value", round(expected_value, 2))
    mlflow.log_metric("treatment_conversion_rate", round(treatment_rate, 4))
    mlflow.log_metric("holdout_conversion_rate", round(holdout_rate, 4))
    mlflow.log_metric("absolute_lift", round(lift, 4))
    mlflow.log_metric("relative_lift_pct", round(lift_pct, 2))
    mlflow.log_metric("n_high_tier", int((scoring_output["tier"] == "High").sum()))
    mlflow.log_metric("n_med_tier", int((scoring_output["tier"] == "Med").sum()))
    mlflow.log_metric("n_low_tier", int((scoring_output["tier"] == "Low").sum()))

    # LOG MONTE CARLO RESULTS per offer
    for offer, r in mc_results.items():
        mlflow.log_metric(f"mc_mean_profit_{offer}", round(r["mean_profit"], 4))
        mlflow.log_metric(f"mc_ci_lower_{offer}", round(r["ci_lower"], 4))
        mlflow.log_metric(f"mc_ci_upper_{offer}", round(r["ci_upper"], 4))
        mlflow.log_metric(f"mc_prob_positive_{offer}", round(r["prob_positive"], 4))

    # LOG THE MODEL ITSELF
    # This saves the trained model as an artifact
    # You can load it later without retraining
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="gradient_boosting_model",
        input_example=X_test.iloc[:3]
    )

    # LOG SCORING OUTPUT as a CSV artifact
    scoring_path = "/tmp/scoring_output.csv"
    scoring_output.to_csv(scoring_path, index=False)
    mlflow.log_artifact(scoring_path, artifact_path="outputs")

    # LOG TAGS — useful for filtering in the MLflow UI
    mlflow.set_tags({
        "domain": "digital_marketing",
        "model_type": "gradient_boosting",
        "pipeline": "propensity_mc_optimisation_ucg",
        "author": "mina_rezaei",
        "budget_scenario": "generous"
    })

    run1_id = mlflow.active_run().info.run_id
    print(f" Run 1 ID: {run1_id}")

print()

# ── RUN 2: Tight budget ($500) ───────────────────────────────────
print("Logging Run 2: tight budget ($500)...")

# Re-run optimisation with tight budget
allocation_tight, ev_tight = optimise_offer_allocation(
    base_probs, budget=500, n_customers=500
)

with mlflow.start_run(run_name="budget_500"):

    mlflow.log_params({
        "n_customers": 5000,
        "test_size": 0.2,
        "n_estimators": 150,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "mc_simulations": 1000,
        "opt_budget": 500,
        "opt_n_targets": 500,
        "holdout_pct": 0.10,
        "random_seed": 42
    })

    # Same model metrics — model didn't change
    mlflow.log_metric("roc_auc", round(auc, 4))
    mlflow.log_metric("pr_auc", round(pr_auc, 4))
    mlflow.log_metric("opt_expected_value", round(ev_tight, 2))
    mlflow.log_metric("absolute_lift", round(lift, 4))
    mlflow.log_metric("relative_lift_pct", round(lift_pct, 2))

    # Log tight budget allocation
    for offer, frac in allocation_tight.items():
        mlflow.log_metric(f"allocation_{offer}", round(frac, 4))

    mlflow.set_tags({
        "domain": "digital_marketing",
        "model_type": "gradient_boosting",
        "pipeline": "propensity_mc_optimisation_ucg",
        "author": "mina_rezaei",
        "budget_scenario": "tight"
    })

    run2_id = mlflow.active_run().info.run_id
    print(f" Run 2 ID: {run2_id}")

print()
print("=" * 55)
print("MLFLOW TRACKING COMPLETE")
print("=" * 55)
print()
print("Both runs logged successfully.")
print()
print("To view in Databricks:")
print(" 1. Click 'Experiments' in the left sidebar")
print(" 2. Search for: customer-offer-optimisation")
print(" 3. Compare the two runs side by side")
print()
print(f" Run 1 (budget $5,000): {run1_id}")
print(f" Run 2 (budget $500): {run2_id}")
print()
print("Pipeline complete ✓")

2026/05/23 22:42:26 INFO mlflow.tracking.fluent: Experiment with name '/customer-offer-optimisation' does not exist. Creating a new experiment.


Logging Run 1: generous budget ($5,000)...


2026/05/23 22:42:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-e74f84d4-60b5.cloud.databricks.com/ml/experiments/3892022708451970/models/m-2af5b029a10b4f708eee25d040411ab7?o=3196592998524725


 Run 1 ID: 3199c8c040c94643a234a0c8b08eb8eb

Logging Run 2: tight budget ($500)...
Expected net value per customer per offer:
 BOGO               $2.138
 Discount_20        $1.510
 Free_Item          $0.794
 Loyalty_Points     $0.833
 Bundle             $1.877

 Run 2 ID: 084dca23dd4b4e1c9bc453401793bf34

MLFLOW TRACKING COMPLETE

Both runs logged successfully.

To view in Databricks:
 1. Click 'Experiments' in the left sidebar
 2. Search for: customer-offer-optimisation
 3. Compare the two runs side by side

 Run 1 (budget $5,000): 3199c8c040c94643a234a0c8b08eb8eb
 Run 2 (budget $500): 084dca23dd4b4e1c9bc453401793bf34

Pipeline complete ✓
